### Evaluating Agent Using LLM-As-A-Judge 

* Note: I'll use Ollama server wrapped in OpenAI for practice.

In [13]:
import os
import json
from pydantic import BaseModel
from dotenv import load_dotenv, find_dotenv
from evals_tools import RetrieverTools  # tools wrapper

from langsmith import Client, wrappers
from langsmith import evaluate
from langsmith import traceable

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

In [2]:
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
_ = load_dotenv(find_dotenv())

**Instantiate Tools**

In [3]:
retriever = RetrieverTools()

@tool
def lexical_search(query: str) -> list:
    """BM25 keyword search with CrossEncoder reranking. Use for exact keyword matches."""
    results = retriever.lexical_search(query)
    return [str(r['document'].page_content) for r in results]

@tool
def dense_search(query: str) -> list:
    """Dense vector search with LLM compression and reranking. Use for semantic search."""
    results = retriever.dense_search(query)
    return [str(r['page_content']) for r in results]

@tool
def graph_search(query: str) -> list:
    """Graph traversal retrieval using metadata relationships. Use for related roles and departments."""
    results = retriever.graph_search(query)
    return [str(doc.page_content) for doc in results]  # serialize to strings

tools = [lexical_search, dense_search, graph_search]
tools_by_name = {tool.name: tool for tool in tools}

Initializing RetrieverTools...
Loaded 1000 documents from '/mnt/c/Users/emman/chicago_employee_data/clean_data/employee_data.csv' 


Setting up BM25 index...
Extracting docs from documents...

Tokenizing corpus for BM25...



Building BM25 index:   0%|          | 0/1000 [00:00<?, ?it/s]

BM25 index ready with 1000 documents... 


Setting up Graph Retriever...
In-memory store created with 1000 enriched documents

All retrievers ready!


In [4]:
llm = ChatOllama(model="qwen3-coder-next:cloud")
llm_with_tools = llm.bind_tools(tools, tool_choice='any')

**Load Ground Truth Dataset**

In [5]:
def groundtruth(data: str):
    with open(data, "r") as f:
        examples = json.load(f)

    print(f"{len(examples)} Groundtruth loaded successfully!")
    return examples

In [6]:
examples = groundtruth("../groundtruth/groundtruth_dataset.json")
print('-'*50, '\n')
print(examples[:5])

9000 Groundtruth loaded successfully!
-------------------------------------------------- 

[{'inputs': {'question': 'Do you know if AARON, JEFFERY M works full-time or part-time?'}, 'outputs': {'answer': {'name': 'AARON, JEFFERY M', 'job_titles': 'LIEUTENANT', 'department': 'CHICAGO POLICE DEPARTMENT', 'full_or_part_time': 'F', 'salary_or_hourly': 'SALARY', 'annual_salary': 165624.0, 'typical_hours': 36.40625, 'hourly_rate': 44.2503125}}, 'metadata': {'difficulty': 'easy', 'source': 'synthetic'}}, {'inputs': {'question': "I'm looking for AARON, JEFFERY M work hours: full-time or part-time?"}, 'outputs': {'answer': {'name': 'AARON, JEFFERY M', 'job_titles': 'LIEUTENANT', 'department': 'CHICAGO POLICE DEPARTMENT', 'full_or_part_time': 'F', 'salary_or_hourly': 'SALARY', 'annual_salary': 165624.0, 'typical_hours': 36.40625, 'hourly_rate': 44.2503125}}, 'metadata': {'difficulty': 'easy', 'source': 'synthetic'}}, {'inputs': {'question': 'Is AARON, JEFFERY M full-time or part-time?'}, 'output

***Fetch Created Groundtruth Dataset from LangSmith***

In [7]:
# Initialize LangSmith client
client = Client()

dataset = client.read_dataset(dataset_name="Chicago Employee Information")
print(f"Dataset loaded: {dataset.name} with id: {dataset.id}")

Dataset loaded: Chicago Employee Information with id: f776372f-12d5-497b-8f3b-26cbb312f2c0


**Agentic Application logic to evaluate**
* Dataset inputs are automatically sent to this output function.

In [8]:
def generate_output(inputs: dict) -> dict:
    """Generate answer using all three retrievers as context."""
    question = inputs["question"]
    
    # Get context from all three retrievers
    lexical_context = lexical_search.invoke({"query": question})
    dense_context = dense_search.invoke({"query": question})
    graph_context = graph_search.invoke({"query": question})
    
    # Combine all contexts
    combined_context = "\n".join([
        "=== Lexical Search Results ===",
        "\n".join(lexical_context),
        "=== Dense Search Results ===",
        "\n".join(dense_context),
        "=== Graph Search Results ===",
        "\n".join(graph_context),
    ])
    
    # Generate answer using LLM with context
    sys_msg = SystemMessage(content=(
        "You are an expert HR data analyst for Chicago city employee records. "
        "Use ONLY the provided context to answer the question. "
        "Return a JSON object with these fields: name, job_titles, department, "
        "full_or_part_time, salary_or_hourly, annual_salary, typical_hours, hourly_rate. "
        "Do NOT invent any data not in the context."
    ))
    
    human_msg = HumanMessage(content=(
        f"Context:\n{combined_context}\n\n"
        f"Question: {question}\n\n"
        "Answer in JSON format only."
    ))
    
    response = llm.invoke([sys_msg, human_msg])
    try:
        parsed = json.loads(response.content)
    except:
        parsed = {"raw": response.content}

    return {"answer": parsed}

In [9]:
# Generate and log Outputs from LLM(Agents App)

# @traceable(name="Chicago Employee App")
def target(inputs):
    return generate_output(inputs)

# Test on 5 examples first
for ex in examples[:5]:
    result = target(ex["inputs"])
    print(f"Question: {ex['inputs']['question']}, \n")
    print(f"Answer: {result['answer']}, \n")
    print('-'*50)


[Lexical Search] Query: 'Do you know if AARON, JEFFERY M works full-time or part-time?'
Searching for: 'Do you know if AARON, JEFFERY M works full-time or part-time?'

Step 1: BM25 retrieval (getting 5 candidates)

Top-3 BM25 results (before reranking):


1. BM25 Score: 14.285
   name: AARON, JEFFERY M
job_titles: LIEUTENANT
department: CHICAGO POLICE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 165624.0
typical_hours: 36.40625
hourly_rate: 44.2503125

2. BM25 Score: 5.256
   name: AMBROZIAK, AARON J
job_titles: FIREFIGHTER/PARAMEDIC
department: CHICAGO FIRE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 122376.0
typical_hours: 36.40625
hourly_rate: 44.2503125

3. BM25 Score: 5.134
   name: ABERO, AARON
job_titles: PARKING ENFORCEMENT AIDE
department: DEPARTMENT OF FINANCE
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 49056.0
typical_hours: 36.40625
hourly_rate: 44.2503125
Step 2: Reranking with CrossEncoder 


 Top-3 


KeyboardInterrupt



**Define an LLM-as-a-judge evaluator to evaluate correctness of the output**

In [10]:
# Point OpenAI client to Ollama's OpenAI-compatible endpoint
ollama_client = wrappers.wrap_openai(
    OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"  
    )
)

In [11]:
def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict):
    evaluator = create_llm_as_judge(
        prompt=CORRECTNESS_PROMPT,
        model="qwen3-coder-next:cloud",  # model string
        client=ollama_client,            # use ollama client
        feedback_key="correctness",
    )
    eval_result = evaluator(
        inputs=inputs, outputs=outputs, reference_outputs=reference_outputs
    )
    return eval_result

**Run experiment**

In [12]:
experiment_results = client.evaluate(
    target,
    data="Chicago Employee Information",
    evaluators=[correctness_evaluator],
    experiment_prefix="experiment-llm-as-a-judge",
    description="Testing LLM Judge system.",
    metadata={"retrievers": "lexical + dense + graph"},
    max_concurrency=2,
)

View the evaluation results for experiment: 'experiment-llm-as-a-judge-f222c450' at:
https://smith.langchain.com/o/f0078a98-c6b7-4bbc-a866-5dd7a9f7a15d/datasets/f776372f-12d5-497b-8f3b-26cbb312f2c0/compare?selectedSessions=a700dbe2-c316-4d83-b69c-136b44b800ef




0it [00:00, ?it/s]


[Lexical Search] Query: 'Please identify What department does ANNERINO, ANTHONY work in?'
Searching for: 'Please identify What department does ANNERINO, ANTHONY work in?'

Step 1: BM25 retrieval (getting 5 candidates)


[Lexical Search] Query: 'Please identify Find the employee with hourly rate 44.2503125'
Searching for: 'Please identify Find the employee with hourly rate 44.2503125'

Step 1: BM25 retrieval (getting 5 candidates)

Top-3 BM25 results (before reranking):


1. BM25 Score: 2.047
   name: AGUILAR, DANIEL A
job_titles: LIBRARY ASSOCIATE - HOURLY
department: CHICAGO PUBLIC LIBRARY
full_or_part_time: P
salary_or_hourly: HOURLY
annual_salary: 111814.14905940594
typical_hours: 20.0
hourly_rate: 32.3

2. BM25 Score: 2.047
   name: AGUIRRE, VICTOR A
job_titles: LIBRARY ASSOCIATE - HOURLY
department: CHICAGO PUBLIC LIBRARY
full_or_part_time: P
salary_or_hourly: HOURLY
annual_salary: 111814.14905940594
typical_hours: 20.0
hourly_rate: 32.3

3. BM25 Score: 2.047
   name: ANTOLIN, AL

Error running evaluator <DynamicRunEvaluator correctness_evaluator> on run 019d73a9-906c-71c3-8efe-a6a72fadde09: TypeError("create_llm_as_judge() got an unexpected keyword argument 'client'")
Traceback (most recent call last):
  File "/mnt/c/Users/emman/chicago_employee_data/.venv/lib/python3.11/site-packages/langsmith/evaluation/_runner.py", line 1637, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/emman/chicago_employee_data/.venv/lib/python3.11/site-packages/langsmith/evaluation/evaluator.py", line 332, in evaluate_run
    result = self.func(
             ^^^^^^^^^^
  File "/mnt/c/Users/emman/chicago_employee_data/.venv/lib/python3.11/site-packages/langsmith/evaluation/evaluator.py", line 758, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_26785/2462127598.py", line 2, in correctness_evalua

Compressed to 0 documents

 Step 3: Reranking compressed documents
--------------------------------------------------

 Top-3 final results:


[Graph Search] Query: 'Can you tell me Which employee earns 111252.0 annually?'

--------------------------------------------------------------------------------
Total Results Found: 5

Results by Depth:
   Depth 0: 1 documents
   Depth 1: 4 documents

Results by Job Category:
   SANITARY: 1 documents
   SENIOR: 1 documents
   GENERAL: 1 documents
   DIR: 1 documents
   LABORER: 1 documents

Results by Department:
   DEPARTMENT OF WATER MANAGEMENT: 2 documents
   DEPARTMENT OF FINANCE: 2 documents
   DEPARTMENT OF STREETS AND SANITATION: 1 documents

--------------------------------------------------------------------------------

 Document 1
   Depth: 0 | Similarity: 0.4927
   Name: ALRUBAYE, ADAM
   Job Title: SANITARY ENGINEER II
   Job Category: SANITARY
   Department: DEPARTMENT OF WATER MANAGEMENT
   Full/Part Time: F

 Document 2
   Depth

...

Compressed to 3 documents
[Lexical Search] Query: 'I'm looking for What department does ALLEN, RYAN M work in?'
Searching for: 'I'm looking for What department does ALLEN, RYAN M work in?'

Step 1: BM25 retrieval (getting 5 candidates)

Top-3 BM25 results (before reranking):


1. BM25 Score: 12.032
   name: ALLEN, RYAN M
job_titles: LIBRARY CLERK
department: CHICAGO PUBLIC LIBRARY
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 77352.0
typical_hours: 36.40625
hourly_rate: 44.2503125

2. BM25 Score: 10.192
   name: ANTONIK, RYAN M
job_titles: LIEUTENANT-EMT
department: CHICAGO FIRE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 151338.0
typical_hours: 36.40625
hourly_rate: 44.2503125

3. BM25 Score: 7.640
   name: ALLEN, KEVIN M
job_titles: SUPVSR OF ACCOUNTING
department: DEPARTMENT OF FINANCE
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 130476.0
typical_hours: 36.40625
hourly_rate: 44.2503125
Step 2: Reranking with CrossEncoder 




Error running target function: Socket operation on non-socket
Traceback (most recent call last):
  File "/mnt/c/Users/emman/chicago_employee_data/.venv/lib/python3.11/site-packages/langsmith/evaluation/_runner.py", line 1939, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/tmp/ipykernel_26785/2089505005.py", line 5, in target
    return generate_output(inputs)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_26785/2664414815.py", line 7, in generate_output
    dense_context = dense_search.invoke({"query": question})
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/emman/chicago_employee_data/.venv/lib/python3.11/site-packages/langchain_core/tools/base.py", line 642, in invoke
    return self.run(tool_input, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/emman/chicago_employee_data/.venv/lib/python3.11/site-packages/langchain_core/tools/base.py", line 1001, in run
    raise error_to_raise
  File "/mn

Compressed to 6 documents
[Lexical Search] Query: 'Who works in the CHICAGO POLICE DEPARTMENT with the title POLICE OFFICER?'
Searching for: 'Who works in the CHICAGO POLICE DEPARTMENT with the title POLICE OFFICER?'

Step 1: BM25 retrieval (getting 5 candidates)

Top-3 BM25 results (before reranking):


1. BM25 Score: 5.101
   name: ALMAZAN, SIDRONIO
job_titles: POLICE OFFICER
department: CHICAGO POLICE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 123576.0
typical_hours: 36.40625
hourly_rate: 44.2503125

2. BM25 Score: 5.101
   name: ALMEDA, BRANDON
job_titles: POLICE OFFICER
department: CHICAGO POLICE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 111252.0
typical_hours: 36.40625
hourly_rate: 44.2503125

3. BM25 Score: 5.101
   name: ALLENSON, SAGE
job_titles: POLICE OFFICER
department: CHICAGO POLICE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 119154.0
typical_hours: 36.40625
hourly_rate: 44.2503125
Step

Error running target function: Socket operation on non-socket
Traceback (most recent call last):
  File "/mnt/c/Users/emman/chicago_employee_data/.venv/lib/python3.11/site-packages/langsmith/evaluation/_runner.py", line 1939, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/tmp/ipykernel_26785/2089505005.py", line 5, in target
    return generate_output(inputs)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_26785/2664414815.py", line 7, in generate_output
    dense_context = dense_search.invoke({"query": question})
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/emman/chicago_employee_data/.venv/lib/python3.11/site-packages/langchain_core/tools/base.py", line 642, in invoke
    return self.run(tool_input, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/emman/chicago_employee_data/.venv/lib/python3.11/site-packages/langchain_core/tools/base.py", line 1001, in run
    raise error_to_raise
  File "/mn

Compressed to 10 documents
[Lexical Search] Query: 'Identify the employee who works in OFFICE OF PUBLIC SAFETY ADMINISTRATION, earns 93504.0, and has the title ASST PAYROLL ADMINISTRATOR.'
Searching for: 'Identify the employee who works in OFFICE OF PUBLIC SAFETY ADMINISTRATION, earns 93504.0, and has the title ASST PAYROLL ADMINISTRATOR.'

Step 1: BM25 retrieval (getting 5 candidates)

Top-3 BM25 results (before reranking):


1. BM25 Score: 33.421
   name: ALVARADO, LISA M
job_titles: ASST PAYROLL ADMINISTRATOR
department: OFFICE OF PUBLIC SAFETY ADMINISTRATION
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 93504.0
typical_hours: 36.40625
hourly_rate: 44.2503125

2. BM25 Score: 15.319
   name: ALEJANDRE, RODRIGO
job_titles: INVESTIGATOR
department: OFFICE OF PUBLIC SAFETY ADMINISTRATION
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 70752.0
typical_hours: 36.40625
hourly_rate: 44.2503125

3. BM25 Score: 14.643
   name: ADUMETEY, REGINALD T
job_titles: DATA 

**Define an LLM-as-a-judge evaluator to evaluate Valid Reasoning of the output**

* 1. Define your evaluator function

In [20]:
# This function receives the inputs and outputs from each test example
def valid_reasoning(inputs: dict, outputs: dict) -> bool:
    """Use an LLM to judge if the reasoning and the answer are consistent."""
    
    instructions = """
Given the following question, answer, and reasoning, determine if the reasoning
is logically valid and consistent with the question and answer.
Respond with ONLY a JSON object like this: {"reasoning_is_valid": true} or {"reasoning_is_valid": false}
Do not include any other text."""

    msg = f"Question: {inputs['question']}\nAnswer: {outputs['answer']}\nReasoning: {outputs['reasoning']}"
    
    response = ollama_client.chat.completions.create(
        model="qwen3-coder-next:cloud",
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": msg}
        ]
    )
    
    try:
        import json, re
        content = response.choices[0].message.content
        clean = re.sub(r"```json|```", "", content).strip()
        parsed = json.loads(clean)
        return parsed.get("reasoning_is_valid", False)  # ← return the boolean score
    except Exception:
        return False

* 2. Define your target function on 5 examples (the application being evaluated). 

In [21]:
def generate_output(inputs: dict) -> dict:
    """Generate answer using all three retrievers as context."""
    question = inputs["question"]
    
    # Get context from all three retrievers
    lexical_context = lexical_search.invoke({"query": question})
    dense_context = dense_search.invoke({"query": question})
    graph_context = graph_search.invoke({"query": question})
    
    # Combine all contexts
    combined_context = "\n".join([
        "=== Lexical Search Results ===",
        "\n".join(lexical_context),
        "=== Dense Search Results ===",
        "\n".join(dense_context),
        "=== Graph Search Results ===",
        "\n".join(graph_context),
    ])
    
    sys_msg = SystemMessage(content=(
        "You are an expert HR data analyst for Chicago city employee records. "
        "Use ONLY the provided context to answer the question. "
        "Return a JSON object with these fields: name, job_titles, department, "
        "full_or_part_time, salary_or_hourly, annual_salary, typical_hours, hourly_rate. "
        "Also include a 'reasoning' field explaining how you arrived at the answer. "
        "Do NOT invent any data not in the context."
    ))
    
    human_msg = HumanMessage(content=(
        f"Context:\n{combined_context}\n\n"
        f"Question: {question}\n\n"
        "Answer in JSON format only with reasoning field included."
    ))
    
    response = llm.invoke([sys_msg, human_msg])
    
    # Parse response to extract answer and reasoning
    try:
        import json, re
        clean = re.sub(r"```json|```", "", response.content).strip()
        parsed = json.loads(clean)
        return {
            "answer": {k: v for k, v in parsed.items() if k != "reasoning"},
            "reasoning": parsed.get("reasoning", "No reasoning provided")
        }
    except Exception:
        return {
            "answer": response.content,
            "reasoning": "Could not parse structured response"
        }


def target(inputs):
    return generate_output(inputs)


# Test on 5 examples first
for ex in examples[:5]:
    result = target(ex["inputs"])
    print(f"Question: {ex['inputs']['question']}, \n")
    print(f"Answer: {result['answer']}, \n")
    print(f"Reasoning: {result['reasoning']}, \n")
    print('-'*50)



[Lexical Search] Query: 'Do you know if AARON, JEFFERY M works full-time or part-time?'
Searching for: 'Do you know if AARON, JEFFERY M works full-time or part-time?'

Step 1: BM25 retrieval (getting 5 candidates)

Top-3 BM25 results (before reranking):


1. BM25 Score: 14.285
   name: AARON, JEFFERY M
job_titles: LIEUTENANT
department: CHICAGO POLICE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 165624.0
typical_hours: 36.40625
hourly_rate: 44.2503125

2. BM25 Score: 5.256
   name: AMBROZIAK, AARON J
job_titles: FIREFIGHTER/PARAMEDIC
department: CHICAGO FIRE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 122376.0
typical_hours: 36.40625
hourly_rate: 44.2503125

3. BM25 Score: 5.134
   name: ABERO, AARON
job_titles: PARKING ENFORCEMENT AIDE
department: DEPARTMENT OF FINANCE
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 49056.0
typical_hours: 36.40625
hourly_rate: 44.2503125
Step 2: Reranking with CrossEncoder 


 Top-3 

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019d73fd-e153-7451-8611-54ae8754f4e9,id=019d73fd-e5b8-7360-845c-8343f2ade5ff; trace=019d73fd-e153-7451-8611-54ae8754f4e9,id=019d73fd-e5b9-75e0-848a-9239d4296eb0; trace=019d73fd-e153-7451-8611-54ae8754f4e9,id=019d73fd-e5b9-75e0-848a-9239d4296eb0; trace=019d73fd-e153-7451-8611-54ae8754f4e9,id=019d73fd-e629-7f52-88a7-0283b1ee9e80; trace=019d73fd-e153-7451-8611-54ae8754f4e9,id=019d73fd-e629-7f52-88a7-029d2014da20; trace=019d73fd-e153-7451-8611-54ae8754f4e9,id=019d73fd-e629-7f52-88a7-029d2014da20; trace=019d73fd-e153-7451-8611-54ae8754f4e9,id=019d73fd-e62a-76d1-a57e-21a47b34e488; trace=019d73fd-a252-7f60-b4d7-c5d41094db25,id=019d73fd-d2f8-7

* 3. Run the evaluation

In [ ]:
# This runs target on each example and applies the valid_reasoning evaluator
results = evaluate(
    target,                                  # Your application function
    data="Chicago Employee Information",     # Dataset to evaluate on
    evaluators=[valid_reasoning],             # List of evaluator functions
    experiment_prefix="experiment-valid-reasoning",
    metadata={"retrievers": "lexical + dense + graph"}
)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019d73fe-539a-74d2-b3d2-5976134f5f6e,id=019d73fe-96c5-7ee2-9db9-089051b45c1f; trace=019d73fe-539a-74d2-b3d2-5976134f5f6e,id=019d73fe-9fae-7370-97ca-6728f27a90b5; trace=019d73fe-539a-74d2-b3d2-5976134f5f6e,id=019d73fe-9fae-7370-97ca-6728f27a90b5; trace=019d73fe-539a-74d2-b3d2-5976134f5f6e,id=019d73fe-96c4-7871-afa5-2edf3ba8b768; trace=019d73fe-539a-74d2-b3d2-5976134f5f6e,id=019d73fe-9fec-77b0-833d-26bcebcf14e9; trace=019d73fe-539a-74d2-b3d2-5976134f5f6e,id=019d73fe-9fee-75a0-ae70-d3938716e47e; trace=019d73fe-539a-74d2-b3d2-5976134f5f6e,id=019d73fe-9fee-75a0-ae70-d3938716e47e; trace=019d73fe-539a-74d2-b3d2-5976134f5f6e,id=019d73fe-9fee-7

View the evaluation results for experiment: 'experiment-valid-reasoning-feea4e36' at:
https://smith.langchain.com/o/f0078a98-c6b7-4bbc-a866-5dd7a9f7a15d/datasets/f776372f-12d5-497b-8f3b-26cbb312f2c0/compare?selectedSessions=3a0a743c-0b93-4512-9426-65f1a710c4a5




0it [00:00, ?it/s]


[Lexical Search] Query: 'Please identify Find the employee with hourly rate 44.2503125'
Searching for: 'Please identify Find the employee with hourly rate 44.2503125'

Step 1: BM25 retrieval (getting 5 candidates)

Top-3 BM25 results (before reranking):


1. BM25 Score: 2.047
   name: AGUILAR, DANIEL A
job_titles: LIBRARY ASSOCIATE - HOURLY
department: CHICAGO PUBLIC LIBRARY
full_or_part_time: P
salary_or_hourly: HOURLY
annual_salary: 111814.14905940594
typical_hours: 20.0
hourly_rate: 32.3

2. BM25 Score: 2.047
   name: AGUIRRE, VICTOR A
job_titles: LIBRARY ASSOCIATE - HOURLY
department: CHICAGO PUBLIC LIBRARY
full_or_part_time: P
salary_or_hourly: HOURLY
annual_salary: 111814.14905940594
typical_hours: 20.0
hourly_rate: 32.3

3. BM25 Score: 2.047
   name: ANTOLIN, ALYSSA
job_titles: LIBRARY CLERK - HOURLY
department: CHICAGO PUBLIC LIBRARY
full_or_part_time: P
salary_or_hourly: HOURLY
annual_salary: 111814.14905940594
typical_hours: 20.0
hourly_rate: 22.82
Step 2: Reranking with Cro

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019d73fe-a642-71e0-97a2-ce21e1ada38e,id=019d73fe-a642-71e0-97a2-ce21e1ada38e; trace=019d73fe-a642-71e0-97a2-ce21e1ada38e,id=019d73fe-a647-7562-941f-9647f9e3b5a2; trace=019d73fe-a642-71e0-97a2-ce21e1ada38e,id=019d73fe-a647-7562-941f-9647f9e3b5a2; trace=019d73fe-a642-71e0-97a2-ce21e1ada38e,id=019d73fe-a7b2-7302-8336-60151d01e41e
Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"tenant exceeded usage limits: Monthly uni